# 03 - Baseline MLP (Keras) e Conversão hls4ml

Este notebook substitui o modelo Ridge por um Multi-Layer Perceptron (MLP) utilizando TensorFlow/Keras. O objetivo é treinar uma rede neural simples compatível com o fluxo do hls4ml para posterior síntese em FPGA.

Etapas:
1. Treinamento da MLP em Keras.
2. Geração do projeto C++ via hls4ml. *(Qualquer OS)*
3. C-Simulation e validação numérica. **(Requer Linux)**
4. Síntese de hardware. **(Requer Linux + Vivado HLS)**

In [11]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Activation
from tensorflow.keras.optimizers import Adam
import hls4ml

sys.path.append(str(Path("..").resolve() / "src"))
from emg_hls4ml_mvp.dataset import build_feature_dataset

## 1. Aquisição e Preparação dos Dados

In [12]:
X, y, _ = build_feature_dataset(
    data_root=Path("../data"),
    subject="Sub001",
    recording_id="Sub001_1_05_450_0",
    target_column="index_z",
)

# Split cronológico: os últimos 20% do tempo servem como conjunto de teste.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Treino: {X_train_scaled.shape} | Teste: {X_test_scaled.shape}")

Treino: (2800, 384) | Teste: (700, 384)


## 2. Treinamento do Modelo MLP

Arquitetura mínima: uma camada oculta de 32 neurônios com ReLU seguida de saída linear escalar.
Esse tamanho é deliberado — quanto menor o modelo, menor o consumo de DSPs e LUTs no FPGA.

In [13]:
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(32, name="fc1"),
    Activation("relu", name="relu1"),
    Dense(1, name="output"),
    Activation("linear", name="linear_out"),
])

model.compile(optimizer=Adam(learning_rate=0.001), loss="mse")
model.summary()

history = model.fit(
    X_train_scaled, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
)

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ fc1 (Dense)                     │ (None, 32)             │        12,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu1 (Activation)              │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            33 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ linear_out (Activation)         │ (None, 1)              │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,353 (48.25 KB)

 Trainable params: 12,353 (48.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3887.7246 - val_loss: 2869.8477
Epoch 2/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 625us/step - loss: 2100.5615 - val_loss: 2051.4219
Epoch 3/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 619us/step - loss: 1842.5530 - val_loss: 1845.1154
Epoch 4/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 606us/step - loss: 1648.3391 - val_loss: 1667.6154
Epoch 5/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 594us/step - loss: 1430.9551 - val_loss: 1458.8575
Epoch 6/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 607us/step - loss: 1195.4796 - val_loss: 1202.3939
Epoch 7/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 599us/step - loss: 956.0363 - val_loss: 996.3553
Epoch 8/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 606us/step - loss: 781.6487 - val_loss: 859.8344
Epoch 9/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 593us/step - loss: 676.5194 - val_loss: 759.6908
Epoch 10/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 611us/step - loss: 625.4908 - val_loss: 676.0229
Epoch 11/30
70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 629us/step - loss: 595.2996 - val_loss: 649.336

In [14]:
keras_pred = model.predict(X_test_scaled).flatten()

mse  = mean_squared_error(y_test, keras_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, keras_pred)
r2   = r2_score(y_test, keras_pred)

print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R2:   {r2:.4f}")

# Curva de loss
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="treino")
plt.plot(history.history["val_loss"], label="validação")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("Curva de Loss")
plt.legend()

# Série temporal: Keras vs Real
plt.subplot(1, 2, 2)
n = min(200, len(y_test))
plt.plot(y_test[:n], label="Real", linewidth=2)
plt.plot(keras_pred[:n], label="Keras (MLP)", linestyle="dashed")
plt.xlabel("Tempo (frames a 100Hz)")
plt.ylabel("Ângulo")
plt.title("Predição Keras vs Real")
plt.legend()

plt.tight_layout()
plt.show()

22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
MSE:  254.3701
RMSE: 15.9490
MAE:  12.5627
R2:   0.9095


<Figure size 1000x400 with 2 Axes>

## 3. Geração do Projeto C++ via hls4ml

Esta célula lê a rede Keras treinada e escreve em disco o código C++ equivalente para síntese em FPGA.
Nenhuma ferramenta da Xilinx é necessária aqui — apenas o hls4ml instalado.

Parâmetros relevantes:
- `Precision: fixed<16,6>` — padrão 16 bits totais, 6 para a parte inteira. Ajustável na seção 4.
- `ReuseFactor: 1` — sem reuso de multiplicadores (máximo paralelismo, máximo recurso).
- `part` — FPGA alvo. Altere conforme a placa disponível no servidor.

In [15]:
config = hls4ml.utils.config_from_keras_model(model, granularity="model")
print("Configuração hls4ml:", config)

hls_model = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    output_dir="model_1/hls4ml_prj",
    part="xczu7ev-ffvc1156-2-e",  # ZCU104; trocar por xc7z020clg484-1 para Zybo Z7
    clock_period=10.0,            # 100 MHz
)

print("Projeto C++ gerado em model_1/hls4ml_prj/")

Configuração hls4ml: {'Model': {'Precision': {'default': 'fixed<16,6>'}, 'ReuseFactor': 1, 'Strategy': 'Latency', 'BramFactor': 1000000000, 'TraceOutput': False}}
Projeto C++ gerado em model_1/hls4ml_prj/


## 4. C-Simulation e Validação Numérica ⚠️ Linux

> **Esta etapa requer ambiente Linux.**
> O `hls_model.compile()` usa o compilador GCC para construir uma biblioteca compartilhada (`.so`) que emula
> a aritmética de ponto fixo do FPGA. No macOS, o Apple Clang atual tem conflito de namespace com os
> cabeçalhos `ap_types` da Xilinx e falha na compilação.

O objetivo desta etapa é medir o **erro de quantização**: diferença entre a predição em ponto flutuante
(Keras, float32) e a predição em ponto fixo (hls4ml, fixed<16,6>).

In [ ]:
# Execute no servidor Linux.
hls_model.compile()

hls_pred = hls_model.predict(X_test_scaled).flatten()

quant_mse = mean_squared_error(keras_pred, hls_pred)
print(f"MSE de quantização (Keras float32 vs HLS fixed<16,6>): {quant_mse:.6f}")

# Visualização: as três curvas sobrepostas
n = min(200, len(y_test))
plt.figure(figsize=(12, 4))
plt.plot(y_test[:n],      label="Real",              linewidth=2)
plt.plot(keras_pred[:n],  label="Keras (float32)",   linestyle="dashed")
plt.plot(hls_pred[:n],    label="HLS (fixed<16,6>)", linestyle="dotted")
plt.xlabel("Tempo (frames a 100Hz)")
plt.ylabel("Ângulo")
plt.title("Real vs Keras vs HLS")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Síntese de Hardware ⚠️ Linux + Vivado HLS

> **Esta etapa requer Linux com Vivado HLS instalado e licenciado.**
> O `hls_model.build()` invoca o Vivado HLS para converter o C++ em RTL (Verilog/VHDL) e
> gerar os relatórios de síntese.

O relatório gerado em `model_1/hls4ml_prj/myproject_prj/solution1/syn/report/` conterá:
- **Latência estimada** (clock cycles)
- **DSP48E** utilizados
- **LUT** e **FF** utilizados
- **BRAM** utilizados

Com esses números, é possível comparar o custo do modelo contra os recursos disponíveis na placa alvo.

In [ ]:
# Execute no servidor Linux com Vivado HLS disponível.
report = hls_model.build(csim=False, synth=True, vsynth=True)
print(report)

## 6. Leitura do Relatório de Síntese

Após o `build`, utilize a célula abaixo para ler o relatório diretamente e registrar os valores
no `docs/development_log.md`.

In [ ]:
# Execute no servidor Linux após o build.
report = hls4ml.report.read_vivado_report("model_1/hls4ml_prj")
hls4ml.report.print_vivado_report(report)